# Klinik Chong Chatbot Ver22 Runtime

Runs the Ver22 Booking FastAPI, chatbot model server and component checks. Ver22 adds interruptible IC prompts, session-based user-name memory and deterministic SQLite-only doctor slot availability retrieval using Malaysia time.


In [ ]:
# ONE-CELL RUNTIME STARTER
# In a fresh Colab runtime, run only this cell.

import json
import os
import re
import sqlite3
import select
import secrets
import subprocess
import sys
import threading
import time
from datetime import date
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "fastapi", "uvicorn", "nest-asyncio", "httpx"
])

from typing import Any, Literal, Optional
import httpx
import nest_asyncio
import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from google.colab import drive
from pydantic import BaseModel, Field

drive.mount("/content/drive")

RUNTIME_FOLDER = (
    "/content/drive/MyDrive/FYP2/"
    "klinik_chong_database"
)
if RUNTIME_FOLDER not in sys.path:
    sys.path.insert(0, RUNTIME_FOLDER)

import importlib
import klinik_chong_runtime
importlib.invalidate_caches()
klinik_chong_runtime = importlib.reload(klinik_chong_runtime)
from klinik_chong_runtime import *

# V22 official clinic public holidays.
# This startup guard is idempotent and protects the persistent SQLite file
# when the API runtime is started from an already-mounted Google Drive session.
PUBLIC_HOLIDAYS_V22 = [
    ("2026-09-16", "Malaysia Day"),
    ("2026-11-06", "Sultan of Perak's Birthday"),
    ("2026-11-08", "Deepavali"),
    ("2026-11-09", "Deepavali"),
    ("2026-12-25", "Christmas"),
    ("2027-01-01", "New Year Holiday"),
]

def ensure_public_holidays_v22():
    database_path = Path(DB_PATH).expanduser().resolve()
    connection = sqlite3.connect(str(database_path), timeout=30)
    try:
        for holiday_date, holiday_name in PUBLIC_HOLIDAYS_V22:
            connection.execute(
                """
                INSERT INTO clinic_holiday (
                    holiday_date,
                    holiday_name,
                    description,
                    is_closed
                )
                VALUES (?, ?, ?, 1)
                ON CONFLICT(holiday_date) DO UPDATE SET
                    holiday_name = excluded.holiday_name,
                    description = excluded.description,
                    is_closed = 1;
                """,
                (
                    holiday_date,
                    holiday_name,
                    "Klinik Chong Public Holiday",
                ),
            )
        connection.commit()

        row = connection.execute(
            """
            SELECT COUNT(*) AS total
            FROM clinic_holiday
            WHERE is_closed = 1
              AND holiday_date IN (?, ?, ?, ?, ?, ?);
            """,
            tuple(item[0] for item in PUBLIC_HOLIDAYS_V22),
        ).fetchone()

        return int(row[0])
    finally:
        connection.close()

ENSURED_PUBLIC_HOLIDAYS_V22 = ensure_public_holidays_v22()
print(
    "V22 official public holidays ensured:",
    ENSURED_PUBLIC_HOLIDAYS_V22,
)

app = FastAPI(
    title="Klinik Chong Booking API",
    description=(
        "Backend API for the Malay-Chinese "
        "chatbot appointment system."
    ),
    version="1.2.0"
)

# Allow chatbot frontend to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.get("/")
async def root():
    return {
        "status": "online",
        "message": "Klinik Chong Booking API"
    }

@app.get("/health")
async def health_check():
    try:
        cursor.execute(
            "SELECT COUNT(*) AS total FROM doctor;"
        )

        total_doctors = cursor.fetchone()["total"]

        return {
            "status": "healthy",
            "database": "connected",
            "total_doctors": total_doctors
        }

    except Exception as error:
        return {
            "status": "unhealthy",
            "database": "disconnected",
            "error": str(error)
        }

print("FastAPI application created successfully.")

@app.get("/doctors")
async def get_doctors_api(
    expertise: str | None = Query(
        default=None,
        description=(
            "Optional expertise such as GENERAL, "
            "OBGYN, PEDIATRICS or DERMATOLOGY."
        )
    )
):
    """
    Return all active doctors, or filter doctors
    according to expertise.
    """

    # If expertise is provided, use existing function
    if expertise:
        result = get_doctors_by_expertise(
            expertise
        )

        if not result["success"]:
            return JSONResponse(
                status_code=400,
                content=result
            )

        return result

    # If expertise is not provided,
    # return all active doctors
    cursor.execute("""
    SELECT
        d_id,
        d_name,
        d_gender,
        d_expertise
    FROM doctor
    WHERE is_active = 1
    ORDER BY d_id;
    """)

    rows = cursor.fetchall()

    doctors = [
        {
            "d_id": row["d_id"],
            "d_name": row["d_name"],
            "d_gender": row["d_gender"],
            "d_expertise": row["d_expertise"]
        }
        for row in rows
    ]

    return {
        "success": True,
        "status": "DOCTORS_FOUND",
        "message": (
            f"{len(doctors)} active doctor(s) found."
        ),
        "expertise": expertise,
        "total_doctors": len(doctors),
        "doctors": doctors
    }

print("GET /doctors endpoint created.")

@app.get("/available-slots")
async def get_available_slots_api(
    d_id: str = Query(
        ...,
        description="Doctor ID, for example D001."
    ),

    appointment_date: str = Query(
        ...,
        description="Date in YYYY-MM-DD format."
    )
):
    """
    Return available appointment slots for
    a selected doctor and date.
    """

    result = get_available_slots(
        d_id=d_id.strip().upper(),
        appointment_date=appointment_date.strip()
    )

    # Invalid user input
    if result["status"] == "INVALID_DATE":
        return JSONResponse(
            status_code=400,
            content=result
        )

    # Doctor does not exist
    if result["status"] == "DOCTOR_NOT_FOUND":
        return JSONResponse(
            status_code=404,
            content=result
        )

    # The following are valid availability results,
    # even though there are no available slots:
    #
    # CLINIC_CLOSED
    # CLINIC_CLOSED_HOLIDAY
    # DOCTOR_OFF_DAY
    # DOCTOR_ON_LEAVE
    # FULLY_BOOKED
    return result

print(
    "GET /available-slots endpoint created."
)

class PatientCreateRequest(BaseModel):
    p_name: str = Field(
        ...,
        min_length=1,
        max_length=120
    )

    p_dob: str = Field(
        ...,
        description="DD-MM-YYYY"
    )

    p_gender: Literal["M", "F"]

    p_ic: str = Field(
        ...,
        description="XXXXXX-XX-XXXX"
    )

    p_contact: str = Field(
        ...,
        description=(
            "01X-XXXXXXX or 01X-XXXXXXXX"
        )
    )

class AppointmentCreateRequest(BaseModel):
    p_id: str
    d_id: str

    appointment_date: str = Field(
        ...,
        description="YYYY-MM-DD"
    )

    start_time: str = Field(
        ...,
        description="HH:MM"
    )

    clinical_note: Optional[str] = Field(
        default=None,
        max_length=720
    )

    status: Literal[
        "PENDING",
        "CONFIRMED"
    ] = "CONFIRMED"

print("API request models created.")

class AppointmentCancelRequest(BaseModel):
    patient_ic: str = Field(
        ...,
        description="XXXXXX-XX-XXXX"
    )

    cancellation_reason: str = Field(
        ...,
        min_length=1,
        max_length=300
    )

print("Cancellation request model created.")
class AppointmentRescheduleRequestV22(BaseModel):
    patient_ic: str = Field(..., description="XXXXXX-XX-XXXX")
    d_id: str = Field(..., min_length=1, max_length=20)
    appointment_date: str = Field(..., description="YYYY-MM-DD")
    start_time: str = Field(..., description="HH:MM")
    clinical_note: Optional[str] = Field(default=None, max_length=720)
    reschedule_reason: str = Field(default="Patient requested reschedule", max_length=300)

class CompleteRebookingRequest(BaseModel):
    cancelled_appointment_id: str
    new_appointment_id: str

    patient_ic: str = Field(
        ...,
        description="XXXXXX-XX-XXXX"
    )

    reschedule_reason: str = Field(
        ...,
        min_length=1,
        max_length=300
    )

print(
    "Complete rebooking request model created."
)

@app.post("/patients")
async def create_patient_api(
    request: PatientCreateRequest
):
    result = create_patient(
        p_name=request.p_name,
        p_dob=request.p_dob,
        p_gender=request.p_gender,
        p_ic=request.p_ic,
        p_contact=request.p_contact
    )

    if result["status"] == "PATIENT_CREATED":
        return JSONResponse(
            status_code=201,
            content=result
        )

    if result["status"] == (
        "PATIENT_ALREADY_EXISTS"
    ):
        return JSONResponse(
            status_code=200,
            content=result
        )

    if result["status"] == "VALIDATION_ERROR":
        return JSONResponse(
            status_code=400,
            content=result
        )

    return JSONResponse(
        status_code=500,
        content=result
    )

print("POST /patients endpoint created.")

@app.post("/appointments")
async def create_appointment_api(
    request: AppointmentCreateRequest
):
    result = create_appointment(
        p_id=request.p_id.strip().upper(),
        d_id=request.d_id.strip().upper(),

        appointment_date=(
            request.appointment_date.strip()
        ),

        start_time=request.start_time.strip(),

        clinical_note=request.clinical_note,
        status=request.status
    )

    if result["status"] == (
        "APPOINTMENT_CREATED"
    ):
        return JSONResponse(
            status_code=201,
            content=result
        )

    if result["status"] in {
        "PATIENT_NOT_FOUND",
        "DOCTOR_NOT_FOUND"
    }:
        return JSONResponse(
            status_code=404,
            content=result
        )

    if result["status"] in {
        "SLOT_UNAVAILABLE",
        "SLOT_ALREADY_BOOKED",
        "FULLY_BOOKED"
    }:
        return JSONResponse(
            status_code=409,
            content=result
        )

    if result["status"] in {
        "INVALID_DATETIME",
        "INVALID_APPOINTMENT_STATUS",
        "CLINICAL_NOTE_TOO_LONG",
        "PAST_APPOINTMENT",
        "CLINIC_CLOSED",
        "CLINIC_CLOSED_HOLIDAY",
        "DOCTOR_OFF_DAY",
        "DOCTOR_ON_LEAVE",
        "DOCTOR_INACTIVE"
    }:
        return JSONResponse(
            status_code=400,
            content=result
        )

    return JSONResponse(
        status_code=500,
        content=result
    )

print("POST /appointments endpoint created.")

@app.get("/appointments/latest")
async def get_latest_appointment_api(
    patient_ic: str = Query(
        ...,
        description="Patient IC for identity verification."
    )
):
    normalized_ic = re.sub(r"[^0-9]", "", patient_ic)
    if not re.fullmatch(r"[0-9]{12}", normalized_ic):
        return JSONResponse(
            status_code=400,
            content={
                "success": False,
                "status": "INVALID_PATIENT_IC",
                "message": "Patient IC must contain 12 digits."
            }
        )

    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    try:
        row = connection.execute("""
            SELECT
                a.appointment_id,
                a.p_id,
                a.status,
                a.appointment_date,
                a.start_time,
                a.end_time,
                a.clinical_note,
                p.p_name,
                p.p_dob,
                p.p_gender,
                p.p_ic,
                p.p_contact,
                d.d_id,
                d.d_name AS doctor_name,
                d.d_expertise AS doctor_expertise
            FROM appointment AS a
            JOIN patient AS p ON p.p_id = a.p_id
            JOIN doctor AS d ON d.d_id = a.d_id
            WHERE REPLACE(p.p_ic, '-', '') = ?
              AND a.status IN ('PENDING', 'CONFIRMED')
            ORDER BY a.appointment_date DESC, a.start_time DESC
            LIMIT 1;
        """, (normalized_ic,)).fetchone()
    finally:
        connection.close()

    if row is None:
        return JSONResponse(
            status_code=404,
            content={
                "success": False,
                "status": "ACTIVE_APPOINTMENT_NOT_FOUND",
                "message": "No active appointment was found for this IC."
            }
        )

    return {
        "success": True,
        "status": "LATEST_ACTIVE_APPOINTMENT_FOUND",
        "appointment": dict(row)
    }

print("GET /appointments/latest endpoint created.")

@app.get(
    "/appointments/{appointment_id}"
)
async def get_appointment_api(
    appointment_id: str,

    patient_ic: str = Query(
        ...,
        description=(
            "Patient IC for identity verification."
        )
    )
):
    result = get_appointment_details(
        appointment_id=(
            appointment_id.strip().upper()
        ),
        patient_ic=patient_ic.strip()
    )

    if result["status"] == (
        "APPOINTMENT_FOUND"
    ):
        return result

    if result["status"] == (
        "APPOINTMENT_NOT_FOUND"
    ):
        return JSONResponse(
            status_code=404,
            content=result
        )

    return JSONResponse(
        status_code=400,
        content=result
    )

print(
    "GET /appointments/{appointment_id} "
    "endpoint created."
)

@app.patch(
    "/appointments/{appointment_id}/cancel"
)
async def cancel_appointment_api(
    appointment_id: str,
    request: AppointmentCancelRequest
):
    result = cancel_appointment(
        appointment_id=(
            appointment_id.strip().upper()
        ),

        patient_ic=request.patient_ic,

        cancellation_reason=(
            request.cancellation_reason
        )
    )

    if result["status"] == (
        "APPOINTMENT_CANCELLED"
    ):
        return result

    if result["status"] == (
        "APPOINTMENT_NOT_FOUND"
    ):
        return JSONResponse(
            status_code=404,
            content=result
        )

    if result["status"] in {
        "ALREADY_CANCELLED",
        "CANCELLATION_NOT_ALLOWED"
    }:
        return JSONResponse(
            status_code=409,
            content=result
        )

    if result["status"] in {
        "INVALID_IC_FORMAT",
        "INVALID_CANCELLATION_REASON",
        "PAST_APPOINTMENT"
    }:
        return JSONResponse(
            status_code=400,
            content=result
        )

    return JSONResponse(
        status_code=500,
        content=result
    )

print(
    "PATCH /appointments/{appointment_id}/cancel "
    "endpoint created."
)

@app.patch("/appointments/{appointment_id}/reschedule")
async def reschedule_appointment_api_v22(
    appointment_id: str,
    request: AppointmentRescheduleRequestV22
):
    result = reschedule_appointment(
        appointment_id=appointment_id.strip().upper(),
        patient_ic=request.patient_ic,
        d_id=request.d_id.strip().upper(),
        appointment_date=request.appointment_date,
        start_time=request.start_time,
        clinical_note=request.clinical_note,
        reschedule_reason=request.reschedule_reason
    )
    if result.get("success"):
        return result
    status = result.get("status")
    if status in {"APPOINTMENT_NOT_FOUND", "PATIENT_NOT_FOUND"}:
        code = 404
    elif status in {"SLOT_ALREADY_BOOKED", "DOCTOR_ON_LEAVE", "RESCHEDULE_NOT_ALLOWED"}:
        code = 409
    elif status in {"INVALID_IC_FORMAT", "INVALID_APPOINTMENT_DATE", "INVALID_START_TIME", "PAST_APPOINTMENT"}:
        code = 400
    else:
        code = 400
    return JSONResponse(status_code=code, content=result)

print("PATCH /appointments/{appointment_id}/reschedule endpoint created.")

@app.post(
    "/appointments/complete-rebooking"
)
async def complete_rebooking_api(
    request: CompleteRebookingRequest
):
    result = (
        complete_rebooking_after_cancel(
            cancelled_appointment_id=(
                request.cancelled_appointment_id
            ),

            new_appointment_id=(
                request.new_appointment_id
            ),

            patient_ic=request.patient_ic,

            reschedule_reason=(
                request.reschedule_reason
            )
        )
    )

    if result["status"] == (
        "REBOOKING_COMPLETED"
    ):
        return result

    if result["status"] in {
        "APPOINTMENT_NOT_FOUND",
        "OLD_APPOINTMENT_NOT_FOUND",
        "NEW_APPOINTMENT_NOT_FOUND"
    }:
        return JSONResponse(
            status_code=404,
            content=result
        )

    if result["status"] in {
        "SAME_APPOINTMENT_ID",
        "OLD_APPOINTMENT_NOT_CANCELLED",
        "PATIENT_MISMATCH",
        "INVALID_NEW_APPOINTMENT",
        "RESCHEDULE_ALREADY_RECORDED"
    }:
        return JSONResponse(
            status_code=409,
            content=result
        )

    if result["status"] in {
        "VALIDATION_ERROR",
        "INVALID_IC_FORMAT",
        "RESCHEDULE_REASON_REQUIRED",
        "RESCHEDULE_REASON_TOO_LONG"
    }:
        return JSONResponse(
            status_code=400,
            content=result
        )

    return JSONResponse(
        status_code=500,
        content=result
    )

print(
    "POST /appointments/complete-rebooking "
    "endpoint created."
)
# V22 Developer Database API
# Kept inside the main API notebook: no extra database-view file required.

DEVELOPER_API_KEY_V22 = "KC-V22-" + secrets.token_urlsafe(24)

class DoctorTakeLeaveRequestV22(BaseModel):
    d_id: str = Field(
        ...,
        min_length=4,
        max_length=20
    )
    leave_date: str = Field(
        ...,
        description="YYYY-MM-DD"
    )
    leave_type: Literal[
        "ANNUAL", "MEDICAL", "EMERGENCY", "OTHER"
    ] = "OTHER"
    reason: Optional[str] = Field(default=None, max_length=300)

def verify_developer_key_v16(
    x_developer_key: str = Header(
        default="",
        alias="X-Developer-Key"
    )
) -> None:
    if x_developer_key != DEVELOPER_API_KEY_V22:
        raise HTTPException(
            status_code=403,
            detail="Invalid Developer View key."
        )

def database_path_v16() -> Path:
    database_path = Path(DB_PATH).expanduser().resolve()

    if not database_path.is_file():
        raise HTTPException(
            status_code=503,
            detail="Klinik Chong database was not found."
        )

    return database_path

def open_developer_database_v16(
    *,
    read_only: bool = True
) -> sqlite3.Connection:
    database_path = database_path_v16()

    if read_only:
        connection = sqlite3.connect(
            f"{database_path.as_uri()}?mode=ro",
            uri=True,
            timeout=10
        )
        connection.execute("PRAGMA query_only = ON;")
    else:
        connection = sqlite3.connect(
            str(database_path),
            timeout=10
        )

    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON;")

    return connection

def developer_table_names_v16(
    connection: sqlite3.Connection
) -> list[str]:
    rows = connection.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name;
    """).fetchall()

    return [
        str(row["name"])
        for row in rows
    ]

def quote_developer_identifier_v16(
    value: str
) -> str:
    return '"' + value.replace('"', '""') + '"'

def developer_json_value_v16(
    value: Any
) -> Any:
    if value is None or isinstance(
        value,
        (str, int, float, bool)
    ):
        return value

    if isinstance(value, bytes):
        return f"<BLOB: {len(value)} bytes>"

    return str(value)

def parse_leave_date_v16(
    value: str
) -> str:
    value = str(value or "").strip()

    try:
        parsed = date.fromisoformat(value)
    except ValueError as error:
        raise HTTPException(
            status_code=400,
            detail=(
                "Take leave date must use "
                "YYYY-MM-DD format."
            )
        ) from error

    if parsed < date.today():
        raise HTTPException(
            status_code=400,
            detail=(
                "Take leave date cannot be "
                "in the past."
            )
        )

    return parsed.isoformat()

# Remove an older Developer Database handler if this
# section is re-run in the same Colab runtime.
developer_paths_v16 = {
    "/developer/database/schema",
    "/developer/database/tables/{table_name}/records",
    "/developer/database/doctor-leave",
}

app.router.routes[:] = [
    route
    for route in app.router.routes
    if getattr(route, "path", None)
    not in developer_paths_v16
]

@app.get("/developer/database/schema")
async def developer_database_schema_v16(
    _: None = Depends(
        verify_developer_key_v16
    )
):
    connection = open_developer_database_v16(
        read_only=True
    )

    try:
        table_names = (
            developer_table_names_v16(
                connection
            )
        )

        tables = []
        relationships = []

        for table_name in table_names:
            quoted_table = (
                quote_developer_identifier_v16(
                    table_name
                )
            )

            column_rows = connection.execute(
                f"PRAGMA table_info({quoted_table});"
            ).fetchall()

            foreign_key_rows = (
                connection.execute(
                    "PRAGMA foreign_key_list"
                    f"({quoted_table});"
                ).fetchall()
            )

            row_count = connection.execute(
                "SELECT COUNT(*) AS total "
                f"FROM {quoted_table};"
            ).fetchone()["total"]

            columns = [
                {
                    "position": row["cid"],
                    "name": row["name"],
                    "type": (
                        row["type"] or "ANY"
                    ),
                    "not_null": bool(
                        row["notnull"]
                    ),
                    "default_value": (
                        row["dflt_value"]
                    ),
                    "primary_key_position": (
                        row["pk"]
                    )
                }
                for row in column_rows
            ]

            foreign_keys = [
                {
                    "id": row["id"],
                    "sequence": row["seq"],
                    "from_column": row["from"],
                    "to_table": row["table"],
                    "to_column": row["to"],
                    "on_update": row["on_update"],
                    "on_delete": row["on_delete"]
                }
                for row in foreign_key_rows
            ]

            for foreign_key in foreign_keys:
                relationships.append({
                    "from_table": table_name,
                    "from_column": (
                        foreign_key[
                            "from_column"
                        ]
                    ),
                    "to_table": (
                        foreign_key["to_table"]
                    ),
                    "to_column": (
                        foreign_key["to_column"]
                    ),
                    "on_update": (
                        foreign_key["on_update"]
                    ),
                    "on_delete": (
                        foreign_key["on_delete"]
                    )
                })

            tables.append({
                "name": table_name,
                "row_count": row_count,
                "columns": columns,
                "foreign_keys": foreign_keys
            })

        holiday_rows = connection.execute("""
            SELECT
                holiday_date,
                holiday_name,
                description,
                is_closed
            FROM clinic_holiday
            WHERE is_closed = 1
            ORDER BY holiday_date;
        """).fetchall()

        public_holidays = [
            {
                "holiday_date": (
                    row["holiday_date"]
                ),
                "holiday_name": (
                    row["holiday_name"]
                ),
                "description": (
                    row["description"]
                ),
                "is_closed": bool(
                    row["is_closed"]
                )
            }
            for row in holiday_rows
        ]

        return {
            "status": "connected",
            "read_only": False,
            "database_name": (
                Path(DB_PATH).name
            ),
            "database_path": str(
                Path(DB_PATH)
            ),
            "table_count": len(tables),
            "tables": tables,
            "relationships": relationships,
            "developer_write_controls": [
                "doctor_leave"
            ],
            "public_holidays": public_holidays
        }

    finally:
        connection.close()

@app.get(
    "/developer/database/tables/"
    "{table_name}/records"
)
async def developer_database_records_v16(
    table_name: str,
    limit: int = Query(
        default=100,
        ge=1,
        le=500
    ),
    offset: int = Query(
        default=0,
        ge=0
    ),
    _: None = Depends(
        verify_developer_key_v16
    )
):
    connection = open_developer_database_v16(
        read_only=True
    )

    try:
        table_names = (
            developer_table_names_v16(
                connection
            )
        )

        if table_name not in table_names:
            raise HTTPException(
                status_code=404,
                detail=(
                    "Database table was "
                    "not found."
                )
            )

        quoted_table = (
            quote_developer_identifier_v16(
                table_name
            )
        )

        total_records = connection.execute(
            "SELECT COUNT(*) AS total "
            f"FROM {quoted_table};"
        ).fetchone()["total"]

        column_rows = connection.execute(
            f"PRAGMA table_info({quoted_table});"
        ).fetchall()

        rows = connection.execute(
            f"SELECT * FROM {quoted_table} "
            "LIMIT ? OFFSET ?;",
            (limit, offset)
        ).fetchall()

        columns = [
            str(row["name"])
            for row in column_rows
        ]

        records = [
            {
                column: (
                    developer_json_value_v16(
                        row[column]
                    )
                )
                for column in columns
            }
            for row in rows
        ]

        return {
            "status": "ok",
            "read_only": True,
            "table": table_name,
            "columns": columns,
            "total_records": total_records,
            "limit": limit,
            "offset": offset,
            "records": records
        }

    finally:
        connection.close()

@app.post(
    "/developer/database/doctor-leave"
)
async def developer_take_doctor_leave_v16(
    request: DoctorTakeLeaveRequestV22,
    _: None = Depends(
        verify_developer_key_v16
    )
):
    doctor_id = request.d_id.strip().upper()
    leave_date = parse_leave_date_v16(
        request.leave_date
    )

    connection = open_developer_database_v16(
        read_only=False
    )

    try:
        doctor = connection.execute("""
            SELECT
                d_id,
                d_name,
                d_expertise,
                is_active
            FROM doctor
            WHERE d_id = ?;
        """, (doctor_id,)).fetchone()

        if doctor is None:
            raise HTTPException(
                status_code=404,
                detail="Doctor was not found."
            )

        if int(doctor["is_active"]) != 1:
            raise HTTPException(
                status_code=400,
                detail=(
                    "Inactive doctor cannot "
                    "be assigned leave."
                )
            )

        active_appointments = (
            connection.execute("""
                SELECT COUNT(*) AS total
                FROM appointment
                WHERE d_id = ?
                  AND appointment_date = ?
                  AND status IN (
                      'PENDING',
                      'CONFIRMED'
                  );
            """, (
                doctor_id,
                leave_date
            )).fetchone()["total"]
        )

        leave_type = request.leave_type.strip().upper()
        reason = (request.reason or "").strip() or (
            f"{leave_type.title()} leave added from V22 Developer Database View."
        )

        connection.execute("""
            INSERT INTO doctor_leave (
                d_id,
                leave_date,
                leave_type,
                reason,
                status
            )
            VALUES (
                ?,
                ?,
                ?,
                ?,
                'APPROVED'
            )
            ON CONFLICT(
                d_id,
                leave_date
            ) DO UPDATE SET
                leave_type = excluded.leave_type,
                reason = excluded.reason,
                status = 'APPROVED';
        """, (
            doctor_id,
            leave_date,
            leave_type,
            reason
        ))

        connection.commit()

        warning = None

        if active_appointments:
            warning = (
                f"{active_appointments} "
                "existing active "
                "appointment(s) already use "
                "this doctor/date and need "
                "manual handling."
            )

        return {
            "success": True,
            "status": (
                "DOCTOR_LEAVE_APPROVED"
            ),
            "message": (
                f"{doctor['d_name']} is on "
                "approved leave on "
                f"{leave_date}."
            ),
            "doctor": {
                "d_id": doctor["d_id"],
                "d_name": doctor["d_name"],
                "d_expertise": (
                    doctor["d_expertise"]
                )
            },
            "leave_date": leave_date,
            "leave_type": leave_type,
            "leave_status": "APPROVED",
            "existing_active_appointments": (
                int(active_appointments)
            ),
            "warning": warning
        }

    except HTTPException:
        connection.rollback()
        raise

    except Exception as error:
        connection.rollback()

        raise HTTPException(
            status_code=500,
            detail=str(error)
        ) from error

    finally:
        connection.close()

# V22 Developer appointment management.
# OVERDUE is calculated from Malaysia local time and is not stored permanently.
with sqlite3.connect(DB_PATH) as migration_connection:
    appointment_columns = {
        row[1]
        for row in migration_connection.execute(
            "PRAGMA table_info(appointment);"
        ).fetchall()
    }
    if "completed_at" not in appointment_columns:
        migration_connection.execute(
            "ALTER TABLE appointment ADD COLUMN completed_at TEXT;"
        )
        migration_connection.commit()

@app.get("/developer/appointments")
async def developer_appointments_v22(
    _: None = Depends(verify_developer_key_v16)
):
    connection = open_developer_database_v16(read_only=True)
    try:
        rows = connection.execute("""
            SELECT
                a.appointment_id,
                CASE
                    WHEN a.status IN ('PENDING', 'CONFIRMED')
                     AND datetime(a.appointment_date || ' ' || a.end_time)
                         < datetime('now', '+8 hours')
                    THEN 'OVERDUE'
                    ELSE a.status
                END AS display_status,
                CASE
                    WHEN a.status IN ('PENDING', 'CONFIRMED')
                     AND datetime(a.appointment_date || ' ' || a.end_time)
                         < datetime('now', '+8 hours')
                    THEN 1 ELSE 0
                END AS is_overdue,
                a.status AS stored_status,
                a.appointment_date,
                a.start_time,
                a.end_time,
                a.clinical_note,
                a.created_at AS appointment_created_at,
                a.updated_at AS appointment_updated_at,
                a.completed_at,
                a.cancelled_at,
                a.cancellation_reason,
                p.p_id,
                p.p_name,
                p.p_dob,
                p.p_gender,
                p.p_ic,
                p.p_contact,
                p.created_at AS patient_created_at,
                p.updated_at AS patient_updated_at,
                d.d_id,
                d.d_name,
                d.d_gender,
                d.d_expertise
            FROM appointment AS a
            JOIN patient AS p ON p.p_id = a.p_id
            JOIN doctor AS d ON d.d_id = a.d_id
            ORDER BY
                CASE WHEN a.status IN ('PENDING', 'CONFIRMED') THEN 0 ELSE 1 END,
                a.appointment_date ASC,
                a.start_time ASC;
        """).fetchall()
        return {
            "status": "ok",
            "timezone": "Asia/Kuala_Lumpur",
            "overdue_is_computed": True,
            "appointments": [dict(row) for row in rows]
        }
    finally:
        connection.close()

@app.post("/developer/appointments/{appointment_id}/complete")
async def developer_complete_appointment_v22(
    appointment_id: str,
    _: None = Depends(verify_developer_key_v16)
):
    normalized_id = appointment_id.strip().upper()
    connection = open_developer_database_v16(read_only=False)
    try:
        appointment = connection.execute(
            "SELECT appointment_id, status FROM appointment WHERE appointment_id = ?;",
            (normalized_id,)
        ).fetchone()
        if appointment is None:
            raise HTTPException(status_code=404, detail="Appointment was not found.")
        if appointment["status"] == "CANCELLED":
            raise HTTPException(status_code=400, detail="A cancelled appointment cannot be completed.")
        if appointment["status"] == "RESCHEDULED":
            raise HTTPException(status_code=400, detail="A rescheduled appointment cannot be completed.")

        connection.execute("""
            UPDATE appointment
            SET status = 'COMPLETED',
                completed_at = CURRENT_TIMESTAMP,
                updated_at = CURRENT_TIMESTAMP
            WHERE appointment_id = ?;
        """, (normalized_id,))
        connection.commit()
        return {
            "success": True,
            "status": "APPOINTMENT_COMPLETED",
            "appointment_id": normalized_id,
            "completed_at": connection.execute(
                "SELECT completed_at FROM appointment WHERE appointment_id = ?;",
                (normalized_id,)
            ).fetchone()["completed_at"]
        }
    except HTTPException:
        connection.rollback()
        raise
    except Exception as error:
        connection.rollback()
        raise HTTPException(status_code=500, detail=str(error)) from error
    finally:
        connection.close()

print("V22 Developer appointment list and completion endpoints created.")

app.openapi_schema = None

print(
    "V22 Developer Database routes "
    "created successfully."
)
print(
    "GET /developer/database/schema "
    "endpoint created."
)
print(
    "GET /developer/database/tables/"
    "{table_name}/records endpoint created."
)
print(
    "POST /developer/database/doctor-leave "
    "endpoint created."
)

# Stop this notebook's previous tunnel/server before restarting them.
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()

if "api_server" in globals():
    api_server.should_exit = True
if "api_thread" in globals() and api_thread.is_alive():
    api_thread.join(timeout=10)

nest_asyncio.apply()
API_HOST = "0.0.0.0"
API_PORT = 8000

def start_api_server():
    global api_server
    config = uvicorn.Config(
        app=app,
        host=API_HOST,
        port=API_PORT,
        log_level="warning"
    )
    api_server = uvicorn.Server(config)
    api_server.run()

api_thread = threading.Thread(
    target=start_api_server,
    daemon=True
)
api_thread.start()

local_health_url = f"http://127.0.0.1:{API_PORT}/health"
for _ in range(20):
    try:
        local_response = httpx.get(local_health_url, timeout=5)
        if local_response.status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Local FastAPI server did not become healthy.")

cloudflared_path = "/content/cloudflared"
if not os.path.exists(cloudflared_path):
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cloudflared_path
    ])
    os.chmod(cloudflared_path, 0o755)

tunnel_process = subprocess.Popen(
    [
        cloudflared_path, "tunnel", "--url",
        f"http://127.0.0.1:{API_PORT}",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

PUBLIC_API_URL = None
tunnel_connected = False
tunnel_deadline = time.time() + 60

# Wait for both the URL and Cloudflare's connection confirmation.
# select() prevents stdout.readline() from blocking forever when no log arrives.
while time.time() < tunnel_deadline:
    if tunnel_process.poll() is not None:
        raise RuntimeError(
            "Cloudflare Tunnel stopped before it became ready."
        )

    readable, _, _ = select.select(
        [tunnel_process.stdout], [], [], 1
    )
    if not readable:
        continue

    line = tunnel_process.stdout.readline()
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )
    if match:
        PUBLIC_API_URL = match.group(0)

    if "Registered tunnel connection" in line:
        tunnel_connected = True

    if PUBLIC_API_URL and tunnel_connected:
        break

if PUBLIC_API_URL is None:
    raise RuntimeError("Cloudflare tunnel URL was not generated.")

print("Cloudflare URL created:", PUBLIC_API_URL)
print("Waiting for the public health check...")

public_api_ready = False
last_public_check = "No response received."

# A new Quick Tunnel can need extra time for DNS and edge routing to warm up.
for attempt in range(1, 46):
    if tunnel_process.poll() is not None:
        last_public_check = "Cloudflare Tunnel process stopped."
        break

    try:
        public_response = httpx.get(
            f"{PUBLIC_API_URL}/health",
            timeout=15,
            follow_redirects=True
        )

        if public_response.status_code == 200:
            public_data = public_response.json()
            if public_data.get("status") == "healthy":
                public_api_ready = True
                break

        last_public_check = (
            f"HTTP {public_response.status_code}: "
            f"{public_response.text[:160]}"
        )

    except (httpx.HTTPError, ValueError) as error:
        last_public_check = (
            f"{type(error).__name__}: {error}"
        )

    if attempt % 5 == 0:
        print(
            f"Still waiting... public check "
            f"{attempt}/45"
        )
    time.sleep(2)

# Save the current Quick Tunnel URL for RunChatbot.ipynb.
API_CONNECTION_PATH = os.path.join(
    RUNTIME_FOLDER,
    "api_connection_v22.json"
)
with open(API_CONNECTION_PATH, "w", encoding="utf-8") as config_file:
    json.dump(
        {
            "version": "V22",
            "api_base_url": PUBLIC_API_URL,
            "developer_api_key": DEVELOPER_API_KEY_V22,
            "public_health_ready": public_api_ready,
            "updated_at_epoch": int(time.time())
        },
        config_file,
        indent=2
    )
print("Shared V22 connection config:", API_CONNECTION_PATH)

print("=" * 60)
if public_api_ready:
    print("KLINIK CHONG API IS READY")
else:
    print("PUBLIC URL CREATED — HEALTH CHECK STILL PENDING")
    print("Last check  :", last_public_check)
    print(
        "Wait 20–30 seconds and open the Health URL below. "
        "The local API is already healthy."
    )

print("Public API :", PUBLIC_API_URL)
print("Health     :", f"{PUBLIC_API_URL}/health")
print("Swagger    :", f"{PUBLIC_API_URL}/docs")
print("ReDoc      :", f"{PUBLIC_API_URL}/redoc")
print("Developer DB:", f"{PUBLIC_API_URL}/developer/database/schema")
print("Doctor leave:", f"{PUBLIC_API_URL}/developer/database/doctor-leave")
print()
print(f'const API_BASE_URL = "{PUBLIC_API_URL}";')
print("=" * 60)

# PART 2 — START V22 CHATBOT SERVER
# ============================================================

from google.colab import drive, userdata
import json
import os
from pathlib import Path
import subprocess
import time
import requests


# ============================================================
# KLINIK CHONG V22 — START SERVER AND VERIFY COMPONENTS
# ============================================================

drive.mount("/content/drive")

FYP2_DIR = Path("/content/drive/MyDrive/FYP2")
CHATBOT_DIR = FYP2_DIR / "chatbot_interface"
API_CONNECTION_PATH = (
    FYP2_DIR / "klinik_chong_database" / "api_connection_v22.json"
)

os.chdir(CHATBOT_DIR)
print("Chatbot directory:", CHATBOT_DIR)

def get_colab_secret(name, required=False):
    """Read a Colab Secret without crashing when an optional key is absent."""
    try:
        value = userdata.get(name)
    except Exception as error:
        if required:
            raise ValueError(
                f"{name} is missing or inaccessible in Colab Secrets."
            ) from error
        return ""
    value = str(value).strip() if value else ""
    if required and not value:
        raise ValueError(f"{name} is missing from Colab Secrets.")
    return value


openai_key = get_colab_secret("OPENAI_API_KEY", required=True)
os.environ["OPENAI_API_KEY"] = openai_key

# V21-compatible Twilio setup:
# - TWILIO_AUTH_TOKEN remains protected in Colab Secrets.
# - Account SID, Twilio caller number and clinic contact retain the working
#   V21 backend defaults inside model_server_ver22.py.
# - Optional Colab Secrets with the same names can override those defaults.
twilio_auth_token = get_colab_secret("TWILIO_AUTH_TOKEN")
twilio_override_names = (
    "TWILIO_ACCOUNT_SID",
    "TWILIO_FROM_NUMBER",
    "EMERGENCY_CONTACT_NUMBER",
)

for name in twilio_override_names:
    override_value = get_colab_secret(name)
    if override_value:
        os.environ[name] = override_value
    else:
        os.environ.pop(name, None)

if twilio_auth_token:
    os.environ["TWILIO_AUTH_TOKEN"] = twilio_auth_token
    print("[✓] Twilio emergency call credentials loaded (V21-compatible)")
else:
    os.environ.pop("TWILIO_AUTH_TOKEN", None)
    print(
        "[!] Twilio emergency-call demo disabled; "
        "TWILIO_AUTH_TOKEN is missing from Colab Secrets."
    )

if not API_CONNECTION_PATH.exists():
    raise FileNotFoundError(
        "The integrated V22 Booking API did not create its connection file."
    )

with API_CONNECTION_PATH.open("r", encoding="utf-8") as config_file:
    api_connection = json.load(config_file)

BOOKING_API_URL = str(api_connection.get("api_base_url", "")).rstrip("/")
DEVELOPER_API_KEY = str(api_connection.get("developer_api_key", ""))

if not BOOKING_API_URL.startswith("https://"):
    raise ValueError("The saved V22 Booking API URL is invalid.")
if not DEVELOPER_API_KEY.startswith("KC-V22-") or len(DEVELOPER_API_KEY) < 20:
    raise ValueError("The saved V22 developer key is invalid.")

try:
    api_health = requests.get(f"{BOOKING_API_URL}/health", timeout=20)
    api_health.raise_for_status()
except requests.RequestException as error:
    raise RuntimeError(
        "The integrated V22 Booking API public URL is not responding. "
        "Rerun this first cell to create a new tunnel."
    ) from error

os.environ["KLINIK_CHONG_BOOKING_API_BASE"] = BOOKING_API_URL
os.environ["KLINIK_CHONG_DEVELOPER_API_KEY"] = DEVELOPER_API_KEY

print("[✓] OpenAI API key loaded")
print("[✓] V22 Booking API connected:", BOOKING_API_URL)
print("[✓] V22 developer key configured")

subprocess.run(
    ["pip", "install", "-q", "-r", "requirements_ver22.txt"],
    check=True
)

subprocess.run(
    "fuser -k 8001/tcp || true",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

LOG_PATH = "/tmp/klinik_chatbot_v22.log"
chatbot_log = open(LOG_PATH, "w", encoding="utf-8")
chatbot_process = subprocess.Popen(
    [
        "python", "model_server_ver22.py",
        "--host", "0.0.0.0",
        "--port", "8001",
        "--booking-api-base", BOOKING_API_URL
    ],
    stdout=chatbot_log,
    stderr=subprocess.STDOUT
)

server_ready = False
for _ in range(60):
    try:
        response = requests.get("http://localhost:8001/api/health", timeout=5)
        if response.status_code == 200:
            server_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if not server_ready:
    chatbot_log.flush()
    print(Path(LOG_PATH).read_text(encoding="utf-8", errors="replace"))
    raise RuntimeError("V22 chatbot server failed to start.")


def check(name, method, path, **kwargs):
    url = f"http://localhost:8001{path}"
    try:
        response = requests.request(method, url, **kwargs)
        passed = 200 <= response.status_code < 300
        print(f"[{'✓' if passed else '✗'}] {name:<29} HTTP {response.status_code}")
        if not passed:
            print("    ", response.text[:500])
        return passed, response
    except requests.RequestException as error:
        print(f"[✗] {name:<29} {error}")
        return False, None


print("\n" + "=" * 68)
print("KLINIK CHONG CHATBOT V22 SYSTEM CHECK")
print("=" * 68)

results = []
results.append(check("Models + server health", "GET", "/api/health?load=true", timeout=180))
results.append(check("Booking API", "GET", "/api/booking-proxy/health", timeout=30))
results.append(check("Doctors database", "GET", "/api/booking-proxy/doctors", timeout=30))
results.append(check("Developer database", "GET", "/api/booking-proxy/developer/database/schema", timeout=30))
results.append(check(
    "Language detection", "POST", "/api/language-detect",
    json={"tokens": ["我", "想", "预约", "医生"], "sentence": "我想预约医生"}, timeout=60
))
results.append(check(
    "XLM-R intention model", "POST", "/api/intention-classify",
    json={"user_input": "我想查看明天下午医生有没有空位"}, timeout=180
))
results.append(check(
    "Emergency detection", "POST", "/api/emergency/detect",
    json={"user_input": "abang saya tak boleh nafas nak call ambulance"}, timeout=60
))
results.append(check(
    "Clinical extraction", "POST", "/api/clinical/extract",
    json={"user_input": "我咳嗽三天了，严重程度五分"}, timeout=90
))
results.append(check(
    "RAG pipeline", "POST", "/api/rag-query",
    json={
        "user_input": "Klinik Chong在哪里？",
        "response_language": "chinese",
        "past_queries": ""
    }, timeout=240
))

passed = sum(1 for ok, _ in results if ok)
print("=" * 68)
print(f"V22 checks passed: {passed}/{len(results)}")
print("Local chatbot:", "http://localhost:8001")
print("Log:", LOG_PATH)
print("=" * 68)

if passed != len(results):
    raise RuntimeError("One or more V22 component checks failed; review the output above.")


In [ ]:
# Install cloudflared if needed
!wget -q \
https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
-O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared

# Open public tunnel for V22 chatbot
!cloudflared tunnel \
    --url http://localhost:8001
